# 04. The consistent attacker: three attackers side by side

The consistent attacker (K) is the constrained attacker (C) with every query repaired into a record a real transaction could produce (`fraudadv.attacks.consistent`). It has its own runs on IEEE-CIS, Sparkov and PaySim. On ULB, TabFormer and BAF no constrained query is an impossible record, so there K equals C. This notebook makes, as CSV files in `reports/tables/`:

- **Table III** (`worst_case_three_attackers.csv`, `worst_case_rank.csv`): worst-case ASR at $\epsilon = 0.25$ under the unconstrained (U), constrained (C) and consistent (K) attackers, with each model's mean normalized rank.
- **Table S-XIII** (`realizable_summary.csv`): share of evasions that are impossible records, from `reports/analysis/consistent/realizable.csv`.
- **Table S-XIV** (`consistent_summary.csv`), **Tables S-XV to S-XVII** (`consistent_grid.csv`), **Table S-XVIII** (`consistent_robust_auc.csv`), **Table S-XIX** (`consistent_history.csv`), **Table S-XX** (`consistent_defenses.csv`).
- **Figure S4** (`reports/fig_transfer_consistent`): transfer rates under the consistent attacker.

**Inputs:** the U and C grid and its stored adversarial examples (`runs/<dataset>/attacks/results*.csv`, `adv_*.npz`, `history*.csv`, `robust_auc*.csv`), the defenses (`runs/<dataset>/defenses/`), the consistent attacker's runs (`runs_consistent/<dataset>/attacks/`, `runs_consistent/<dataset>/defenses/`), and the processed data (`data/processed/<dataset>`), on which the realizability check runs.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

## Which evasions are impossible records (the input of Table S-XIII)

`fraudadv.report.realizable_share` applies the realizability check (`fraudadv.attacks.realizable.violations_all`: the hand-written rules plus the structural zeros learned from the training data) to every test record and to the successful high-confidence white-box evasions ($\epsilon = 0.5$) that the grid stored for each attacker. An evasion counts as impossible only if it breaks a rule its original record did not. It writes `reports/analysis/consistent/realizable.csv` and takes about 10 seconds. Below: the share of impossible evasions per dataset and attacker (minimum, maximum and median over models, as the module prints it), then the first rows of the CSV.

In [ ]:
(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)     # the output folders, as make_results.sh makes them

import contextlib
import io

from fraudadv.report import consistent_tables as CT
from fraudadv.report import realizable_share as RS

printed = io.StringIO()
with contextlib.redirect_stdout(printed):          # the module prints every row, then the summary shown here
    RS.main()                                      # reports/analysis/consistent/realizable.csv
lines = printed.getvalue().splitlines()
start = next(k for k, line in enumerate(lines) if line.split() == ["min", "max", "median"])
print("share of successful evasions that are impossible records, over models:")
print("\n".join(lines[start:]))

In [ ]:
nb.show_csv("reports/analysis/consistent/realizable.csv", rows=20);

## The three attackers in one grid

`consistent_tables.grid_three_attackers` joins the study's U and C grid (`make_report.load_attacks`) with the consistent attacker's grid (`runs_consistent/<dataset>/attacks/results_consistent_*.csv`). On ULB, TabFormer and BAF the K rows are copies of the C rows. Every table below starts from this frame.

In [ ]:
G = CT.grid_three_attackers()
G.groupby(["dataset", "mode"]).size().unstack()

## Table III: worst case under the three attackers

`consistent_tables.worst_case` writes `worst_case_three_attackers.csv`: the worst case over all attacks at $\epsilon = 0.25$ for every model, dataset and attacker (K only where it differs from C), and `worst_case_rank.csv`: each model's mean normalized rank over the six consistent columns (K, or C where they coincide; lower is more robust). Below, the table as one row per model.

In [ ]:
W = CT.worst_case(G)            # reports/tables/worst_case_three_attackers.csv, worst_case_rank.csv
R = nb.show_csv("reports/tables/worst_case_rank.csv")
W.pivot_table(index="model", columns=["dataset", "attacker"], values="asr").round(3)

## Table S-XIII: impossible records

`consistent_tables.realizable` writes `realizable_summary.csv` from `realizable.csv`: per dataset and attacker, the range over models of the share of evasions that are impossible records, and the check's rate on clean test records (its false-positive cost).

In [ ]:
CT.realizable()                 # reports/tables/realizable_summary.csv
nb.show_csv("reports/tables/realizable_summary.csv");

## Table S-XIV: the consistent attacker in brief

`consistent_tables.summary` writes `consistent_summary.csv`: C beside K at $\epsilon = 0.25$ per attack (mean over the models the attack applies to), history injection at $k = 8$, and each model's best defense.

In [ ]:
S = CT.summary(G)               # reports/tables/consistent_summary.csv
S.pivot_table(index="result", columns=["dataset", "attacker"], values="mean_asr", sort=False).round(3)

## Tables S-XV to S-XVII: the consistent grid per dataset

`consistent_tables.grid` writes `consistent_grid.csv`: on each K dataset, every attack, model and budget, C beside K, with the robust AUC and KS at $\epsilon = 0.25$. Shown here: PaySim at $\epsilon = 0.25$.

In [ ]:
K = CT.grid(G)                  # reports/tables/consistent_grid.csv
K[(K["dataset"] == "paysim") & (K["eps"] == 0.25)].pivot_table(index=["attack", "model"], columns="attacker", values="asr").round(3)

## Table S-XVIII: ranking metrics under the consistent attacker

`consistent_tables.robust_auc` writes `consistent_robust_auc.csv`: the change in AUC and KS (unit- and dollar-weighted) under the score-minimizing attacker at $\epsilon = 0.25$, C beside K, from `robust_auc*.csv` of both attackers.

In [ ]:
CT.robust_auc()                 # reports/tables/consistent_robust_auc.csv
nb.show_csv("reports/tables/consistent_robust_auc.csv", rows=20);

## Table S-XIX: history injection with realizable warm-up records

`consistent_tables.history` writes `consistent_history.csv`: the ASR of history injection after $k$ injected records, with the study's records (S) beside the repaired ones (K), and how many frauds needed a repair.

In [ ]:
CT.history()                    # reports/tables/consistent_history.csv
nb.show_csv("reports/tables/consistent_history.csv", rows=20);

## Table S-XX: defenses under the consistent attacker

`consistent_tables.defenses` writes `consistent_defenses.csv`: the defended worst case at $\epsilon = 0.25$ of the tabular models, C beside K, per defense.

In [ ]:
D = CT.defenses()               # reports/tables/consistent_defenses.csv
D.pivot_table(index=["dataset", "model"], columns="defense", values=["asr_constrained", "asr_consistent"]).round(3)

## Figure S4: transfer under the consistent attacker

`consistent_tables.figure_transfer` draws the transfer figure of `make_report.fig_transfer` from the consistent attacker's `runs_consistent/<dataset>/attacks/transfer_consistent.npz` on the three K datasets, in a scratch folder, and copies it to `reports/fig_transfer_consistent.{pdf,png}`.

In [ ]:
CT.figure_transfer()            # reports/fig_transfer_consistent.{pdf,png}
nb.show_figure("fig_transfer_consistent")

## Run it yourself: constrained and consistent projections of perturbed frauds

This cell takes 20 random test frauds each of Sparkov and PaySim and gives each one the same random step inside the budget $\epsilon = 0.25$. It then projects the step onto the constrained threat model and onto the consistent one (`Constraints(meta, X_train, mode).project`, as every attack step does). The realizability check (`fraudadv.attacks.realizable.violations_all`) is applied to the originals and to both projections.

Most constrained rows are impossible records, for example a Sparkov merchant and distance that moved independently, so that the distance no longer matches its definition, or a PaySim payee history that cannot occur. Every consistent row is realizable. The consistent rows still move, within the same budget (largest L-inf move). It takes a few seconds and writes nothing. These are random steps on 40 frauds, an illustration of the threat models, not the study's evasions (those are counted in Table S-XIII above).

In [ ]:
import numpy as np
import pandas as pd

from fraudadv.attacks.constraints import Constraints
from fraudadv.attacks.realizable import violations_all
from fraudadv.data.load import load

EPS, N = 0.25, 20
rows = []
for d in ("sparkov", "paysim"):
    a, meta = load(d)
    frauds = np.flatnonzero(a["y_test"] == 1)
    X0 = a["X_test"][np.sort(np.random.default_rng(0).choice(frauds, N, replace=False))]
    step = np.random.default_rng(1).uniform(-EPS, EPS, X0.shape).astype(np.float32)   # one random step in the budget
    constrained = Constraints(meta, a["X_train"], "constrained")
    consistent = Constraints(meta, a["X_train"], "consistent")
    X_c = constrained.project(X0 + step, X0, EPS)
    X_k = consistent.project(X0 + step, X0, EPS)
    impossible_before = violations_all(d, X0)
    for name, X in (("constrained", X_c), ("consistent", X_k)):
        impossible = violations_all(d, X) & ~impossible_before      # breaks a rule its original record did not
        rows.append({"dataset": d, "threat model": name, "frauds": N, "impossible originals": int(impossible_before.sum()),
                     "rows moved": int((np.abs(X - X0).max(1) > 0).sum()), "largest L-inf move": float(constrained.linf(X, X0).max()),
                     "impossible after projection": int(impossible.sum()), "realizable after projection": int((~impossible).sum())})
pd.DataFrame(rows)

## Rerun the runs behind these results from scratch

The cell below holds the calls of the `consistent` stage of `scripts/reproduce.sh`, with the study's options: the consistent attack grid, its transfer matrix, the ranking metrics at $\epsilon = 0.25$, history injection with repaired warm-up records, and the defenses. The stage exists only for IEEE-CIS, Sparkov and PaySim (elsewhere K equals C), so the example dataset here is Sparkov, not ULB. It reads the study's grid (`runs/<dataset>/attacks/`: thresholds and fraud sample) and needs it first (notebook 03). This takes hours of CPU per dataset. It writes into `runs_consistent/<dataset>/` and replaces the saved results, so it is off here (`RERUN = False`). Afterwards, run this notebook again from the top.

In [ ]:
RERUN = False
DATASET = "sparkov"        # ieee_cis | sparkov | paysim: the datasets with a separate consistent arm

if RERUN:
    from fraudadv.attacks import history_consistent, robust_auc, run_attacks, transfer
    from fraudadv.defenses import run_defenses_consistent

    ALL = ["xgboost", "catboost", "mlp", "ft_transformer", "ivis", "hgnn", "hgt"]   # tabular and graph families
    FMS = ["chronos2", "timesfm"]                                                   # foundation models
    import chronos  # noqa: F401  must be imported before TensorFlow (ivis loads it): the reverse order aborts
    run_attacks.run(DATASET, ALL, modes=("consistent",), full_bb=True, milp_budget=3600, tag="_consistent_all")
    run_attacks.run(DATASET, FMS, modes=("consistent",), full_bb=True, milp_budget=3600,
                    fm_full_n=100000 if DATASET == "sparkov" else 240, tag="_consistent_fm")
    transfer.run(DATASET, modes=("consistent",))
    robust_auc.run(DATASET, ALL, tag="_consistent_all", modes=("consistent",), eps=[0.25])
    robust_auc.run(DATASET, FMS, tag="_consistent_fm", modes=("consistent",), eps=[0.25])
    if DATASET in ("sparkov", "ieee_cis"):                                         # history injection needs card histories
        history_consistent.run(DATASET, ALL)
        for fam in FMS:
            history_consistent.run(DATASET, [fam], tag=f"_fm_{fam}")
    run_defenses_consistent.run(DATASET, run_defenses_consistent.FAMILIES.split(","))

## Check against the reference

Every file this notebook wrote, compared byte for byte with the reference build (the folder in `FRAUDADV_REFERENCE`, holding a previous build of `reports/`).

In [ ]:
nb.check(["reports/analysis/consistent/realizable.csv"]
         + [f"reports/tables/{name}.csv" for name in ("worst_case_three_attackers", "worst_case_rank", "realizable_summary",
                                                      "consistent_summary", "consistent_grid", "consistent_robust_auc",
                                                      "consistent_history", "consistent_defenses")]
         + [f"reports/fig_transfer_consistent.{ext}" for ext in ("pdf", "png")]);